# 04 · hierarchical Gaussian — smooth time prior, residual spread as the quantile

The approach as specified:

1. **day of week and time of day give the prior**, smooth unless the data proves
   otherwise;
2. **recent usage shifts the prediction off that prior**;
3. **the residual spread turns a mean into a quantile**, assuming normality.

### Why ridge rather than a sampler

With a Gaussian likelihood and Gaussian priors on the coefficients, the posterior mean
**is** the ridge solution — `alpha` is the prior precision. The smoothness the priors
were meant to impose is carried by the harmonic basis: a shrunk coefficient on the
third harmonic *is* "smooth unless the data proves otherwise". A sampler would buy
credible intervals this stage never reads, at a dependency cost the tick cannot take.

### The assumption, left checkable

`reserve = mean + z·σ` is only a true quantile if the residuals are normal. They are
not: the target is bounded at 0, spikes at exactly 0, and is bounded at 100. So `z` is
tuned by **cost** rather than read off a normal table, and the gap between the tuned
`z` and the normal `z` for `q = 0.909` (= 1.335) is the price of the assumption.

One σ per state is also tried, since a quiet night and a working afternoon plainly do
not share a spread.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
import dataset as D

frame = D.build("claude")
train, validation, test = D.split(frame)
COLS = D.features(frame)
print("\ncost model:  stolen $%.2f/pct   wasted $%.2f/pct   ratio %.0f:1"
      % (D.COST_STOLEN, D.COST_WASTED, D.COST_STOLEN / D.COST_WASTED))
print("the cost-optimal quantile is therefore %.4f" % D.optimal_quantile())
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
print("normal z for q=%.3f is %.3f" % (D.optimal_quantile(), norm.ppf(D.optimal_quantile())))

12075 anchors over 41.9 days, embargo 300 min at each seam
  train        7185 rows  08-24 17:55 -> 09-18 16:35
  validation   2355 rows  09-18 21:40 -> 09-27 01:50
  test         2415 rows  09-27 06:55 -> 10-05 16:05
  dropped to the embargo: 120

cost model:  stolen $3.20/pct   wasted $0.32/pct   ratio 10:1
the cost-optimal quantile is therefore 0.9091
normal z for q=0.909 is 1.335


## The objective

Not accuracy — **money**. Two unit costs per percent of a 5-hour meter:

| | $/pct | Meaning |
|---|---|---|
| `COST_WASTED` | **0.32** | quota the bot was refused although the human never came — exactly 1% of a $32 unit |
| `COST_STOLEN` | **3.20** | quota the human wanted and the bot took — 10×, because the user has first claim |

Hyper-parameters below are chosen on **validation**, by cost. The test set is read
only in `05_comparison.ipynb`.

In [2]:
def fit_gauss(fit_frame, N, alpha, per_state):
    m = ~np.isnan(fit_frame["y%d" % N].values)
    pipe = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    pipe.fit(fit_frame.loc[m, COLS], fit_frame.loc[m, "y%d" % N])
    resid = (fit_frame.loc[m, "y%d" % N] - pipe.predict(fit_frame.loc[m, COLS])).values
    if per_state:
        s = D.state(fit_frame.loc[m])
        sigma = {st: float(np.std(resid[s == st])) if (s == st).sum() > 20
                 else float(np.std(resid)) for st in D.STATES}
    else:
        sigma = {st: float(np.std(resid)) for st in D.STATES}
    return pipe, sigma

def predict_gauss(model, f, z):
    pipe, sigma = model
    return pipe.predict(f[COLS]) + z * np.array([sigma[st] for st in D.state(f)])

cache, candidates = {}, {}
for alpha in (1.0, 10.0, 100.0, 1000.0):
    for per_state in (False, True):
        for N in D.HORIZONS:
            cache[(alpha, per_state, N)] = fit_gauss(train, N, alpha, per_state)
        for z in (0.5, 1.0, 1.335, 1.645, 2.0, 2.5, 3.0):
            candidates["a=%g %s z=%.3f" % (alpha, "sig/state" if per_state else "sig", z)] = \
                (alpha, per_state, z)
print("fitted %d ridges, %d candidate settings" % (len(cache), len(candidates)))

fitted 40 ridges, 56 candidate settings


In [3]:
chosen, table = D.tune("hierarchical gaussian", candidates, validation,
    lambda c, N: predict_gauss(cache[(c[0], c[1], N)], validation, c[2]), show=4)

N = 30   best: a=1000 sig/state z=0.500     $0.4825   (worst $3.6250)
                  config  cost_usd  stolen_usd  wasted_usd
a=1000 sig/state z=0.500    0.4825      0.4454      0.0371
      a=1000 sig z=1.000    0.4943      0.4609      0.0335
a=1000 sig/state z=1.000    0.4965      0.2593      0.2372
 a=100 sig/state z=0.500    0.5103      0.4655      0.0448

N = 60   best: a=1000 sig/state z=0.500     $1.2736   (worst $7.3524)
                  config  cost_usd  stolen_usd  wasted_usd
a=1000 sig/state z=0.500    1.2736      1.0744      0.1992
 a=100 sig/state z=0.500    1.3899      1.1360      0.2539
  a=10 sig/state z=0.500    1.4162      1.1472      0.2690
   a=1 sig/state z=0.500    1.4180      1.1467      0.2713

N = 120  best: a=1000 sig/state z=0.500     $3.4727   (worst $12.3398)
                  config  cost_usd  stolen_usd  wasted_usd
a=1000 sig/state z=0.500    3.4727      2.5192      0.9535
 a=100 sig/state z=0.500    3.5966      2.6003      0.9963
  a=10 sig/state z=0

In [4]:
t = table.copy()
t["z"] = [float(s.split("z=")[1]) for s in t["config"]]
print("cheapest cost per z, per horizon  (normal z for q=0.909 is 1.335)")
print(t.groupby(["N", "z"])["cost_usd"].min().unstack().round(3).to_string())

cheapest cost per z, per horizon  (normal z for q=0.909 is 1.335)
z    0.500  1.000   1.335   1.645   2.000   2.500   3.000
N                                                        
30   0.482  0.494   0.550   0.782   1.217   2.115   3.023
60   1.274  1.428   2.131   2.949   3.920   5.319   6.794
120  3.473  4.267   5.250   6.255   7.545   9.589  11.866
180  5.729  6.582   7.664   8.872  10.362  12.785  15.438
300  9.006  9.693  10.940  12.365  14.400  17.497  20.479


In [5]:
# How badly does the normal assumption fail? Compare the empirical residual
# quantiles against what a normal of the same sigma would predict.
alpha, per_state, z = chosen[300][1]
pipe, sigma = cache[(alpha, per_state, 300)]
m = ~np.isnan(train["y300"].values)
resid = (train.loc[m, "y300"] - pipe.predict(train.loc[m, COLS])).values
s = float(np.std(resid))
print("sigma = %.1f   residual quantile vs normal(0, sigma)" % s)
for q in (50, 75, 90, 90.9, 95, 99):
    print("  p%-5.1f  empirical %7.1f    normal %7.1f" % (q, np.percentile(resid, q),
                                                          norm.ppf(q / 100) * s))

sigma = 24.1   residual quantile vs normal(0, sigma)
  p50.0   empirical    -4.3    normal     0.0
  p75.0   empirical     4.5    normal    16.3
  p90.0   empirical    26.0    normal    30.9
  p90.9   empirical    32.0    normal    32.2
  p95.0   empirical    59.9    normal    39.6
  p99.0   empirical    87.1    normal    56.1


In [6]:
val = [D.evaluate("hierarchical gaussian", validation["y%d" % N].values,
                  predict_gauss(cache[(chosen[N][1][0], chosen[N][1][1], N)],
                                validation, chosen[N][1][2])) | {"N": N}
       for N in D.HORIZONS]
print(pd.DataFrame(val).set_index("N")[["cost_usd", "stolen", "wasted", "coverage"]].round(3))
print()
print(D.by_cell(validation, validation["y300"].values,
                {"gaussian": predict_gauss(cache[(alpha, per_state, 300)], validation, z)}
                ).to_string(index=False))
D.save("04_hierarchical_gaussian",
       {"kind": "gaussian_ridge",
        "per_horizon": {str(N): {"alpha": chosen[N][1][0],
                                 "sigma_per_state": bool(chosen[N][1][1]),
                                 "z": chosen[N][1][2]} for N in D.HORIZONS}}, val)

     cost_usd  stolen  wasted  coverage
N                                      
30      0.482   0.139   0.116    97.197
60      1.274   0.336   0.623    95.924
120     3.473   0.787   2.980    94.820
180     5.729   1.237   5.531    93.418
300     9.006   1.794  10.207    91.932

 state      band   n  ACTUAL p95  gaussian
  idle     night 759         5.0      12.6
  idle   morning 421        38.0      22.8
  idle afternoon 253        40.0      35.6
  idle   evening 479         0.0      17.5
  warm     night  20         3.0      19.4
  warm   morning  28        83.0      28.3
  warm afternoon  62        84.9      39.3
  warm   evening  92         0.0      29.0
active     night  12         3.0      21.9
active   morning  31        79.0      42.4
active afternoon 165        84.0      46.8
active   evening  33         5.4      36.4
froze 04_hierarchical_gaussian -> design/02_prediction/lab/results/04_hierarchical_gaussian.json


PosixPath('/Users/jeanlescut/dev/agent-quota-maximizer/design/02_prediction/lab/results/04_hierarchical_gaussian.json')